# End-to-End Ontology Alignment: OBO + OWL + SSSOM

This tutorial demonstrates how to align two disease ontologies using three complementary data sources:

1. **OBO file** — hierarchy of one ontology (MONDO disease terms)
2. **OWL file** — hierarchy of another ontology (ORDO rare disease terms)
3. **SSSOM file** — candidate cross-ontology mappings with confidence scores

We will:
- Convert each source to a boomer KB
- Merge them into a single KB
- Solve to find the most probable consistent alignment

All steps work from the CLI — no Python code required.

## Step 0: Create Example Files

First, let’s create three small files representing our data sources.

In [1]:
%%bash
# MONDO hierarchy (OBO format)
cat > /tmp/mondo_subset.obo << 'EOF'
format-version: 1.4
ontology: mondo-subset

[Term]
id: MONDO:0000001
name: disease

[Term]
id: MONDO:0001234
name: alpha disease
is_a: MONDO:0000001 ! disease
xref: ORDO:100

[Term]
id: MONDO:0005678
name: beta disease
is_a: MONDO:0000001 ! disease
xref: ORDO:200

[Term]
id: MONDO:0009999
name: gamma disease
is_a: MONDO:0000001 ! disease
EOF

echo "Created MONDO OBO file"
cat /tmp/mondo_subset.obo

Created MONDO OBO file


format-version: 1.4
ontology: mondo-subset

[Term]
id: MONDO:0000001
name: disease

[Term]
id: MONDO

:0001234
name: alpha disease
is_a: MONDO:0000001 ! disease
xref: ORDO:100

[Term]
id: MONDO:0005678


name: beta disease
is_a: MONDO:0000001 ! disease
xref: ORDO:200

[Term]
id: MONDO:0009999
name: gamm

a disease
is_a: MONDO:0000001 ! disease


In [2]:
%%bash
# ORDO hierarchy (OWL Functional Syntax)
cat > /tmp/ordo_subset.ofn << 'EOF'
Prefix(ORDO:=<http://www.orpha.net/ORDO/Orphanet_>)
Prefix(owl:=<http://www.w3.org/2002/07/owl#>)
Prefix(rdfs:=<http://www.w3.org/2000/01/rdf-schema#>)

Ontology(<http://www.orpha.net/ORDO/ordo-subset>

Declaration(Class(ORDO:100))
Declaration(Class(ORDO:200))
Declaration(Class(ORDO:300))
Declaration(Class(ORDO:999))

SubClassOf(ORDO:100 ORDO:999)
SubClassOf(ORDO:200 ORDO:999)
SubClassOf(ORDO:300 ORDO:999)

DisjointClasses(ORDO:100 ORDO:200 ORDO:300)

AnnotationAssertion(rdfs:label ORDO:100 "Alpha rare disease")
AnnotationAssertion(rdfs:label ORDO:200 "Beta rare disease")
AnnotationAssertion(rdfs:label ORDO:300 "Delta rare disease")
AnnotationAssertion(rdfs:label ORDO:999 "Rare disease grouping")
)
EOF

echo "Created ORDO OWL file"
cat /tmp/ordo_subset.ofn

Created ORDO OWL file


Prefix(ORDO:=<http://www.orpha.net/ORDO/Orphanet_>)
Prefix(owl:=<http://www.w3.org/2002/07/owl#>)
Pr

efix(rdfs:=<http://www.w3.org/2000/01/rdf-schema#>)

Ontology(<http://www.orpha.net/ORDO/ordo-subset

>

Declaration(Class(ORDO:100))
Declaration(Class(ORDO:200))
Declaration(Class(ORDO:300))
Declaratio

n(Class(ORDO:999))

SubClassOf(ORDO:100 ORDO:999)
SubClassOf(ORDO:200 ORDO:999)
SubClassOf(ORDO:300 

ORDO:999)

DisjointClasses(ORDO:100 ORDO:200 ORDO:300)

AnnotationAssertion(rdfs:label ORDO:100 "Alp

ha rare disease")
AnnotationAssertion(rdfs:label ORDO:200 "Beta rare disease")
AnnotationAssertion(r

dfs:label ORDO:300 "Delta rare disease")
AnnotationAssertion(rdfs:label ORDO:999 "Rare disease group

ing")
)


In [3]:
%%bash
# Cross-ontology mappings (SSSOM TSV)
cat > /tmp/mondo_ordo_mappings.sssom.tsv << 'SEOF'
#curie_map:
#  MONDO: http://purl.obolibrary.org/obo/MONDO_
#  ORDO: http://www.orpha.net/ORDO/Orphanet_
#  skos: http://www.w3.org/2004/02/skos/core#
#  semapv: https://w3id.org/semapv/vocab/
#mapping_set_id: https://example.org/mondo-ordo-mappings
#mapping_set_description: MONDO to ORDO candidate mappings
subject_id	subject_label	predicate_id	object_id	object_label	mapping_justification	confidence
MONDO:0001234	alpha disease	skos:exactMatch	ORDO:100	Alpha rare disease	semapv:LexicalMatching	0.9
MONDO:0005678	beta disease	skos:exactMatch	ORDO:200	Beta rare disease	semapv:LexicalMatching	0.85
MONDO:0009999	gamma disease	skos:exactMatch	ORDO:300	Delta rare disease	semapv:LexicalMatching	0.6
MONDO:0009999	gamma disease	skos:exactMatch	ORDO:100	Alpha rare disease	semapv:LexicalMatching	0.3
SEOF

echo "Created SSSOM mappings file"
cat /tmp/mondo_ordo_mappings.sssom.tsv

Created SSSOM mappings file


#curie_map:
#  MONDO: http://purl.obolibrary.org/obo/MONDO_
#  ORDO: http://www.orpha.net/ORDO/Orpha

net_
#  skos: http://www.w3.org/2004/02/skos/core#
#  semapv: https://w3id.org/semapv/vocab/
#mappin

g_set_id: https://example.org/mondo-ordo-mappings
#mapping_set_description: MONDO to ORDO candidate 

mappings
subject_id	subject_label	predicate_id	object_id	object_label	mapping_justification	confiden

ce
MONDO:0001234	alpha disease	skos:exactMatch	ORDO:100	Alpha rare disease	semapv:LexicalMatching	0.

9
MONDO:0005678	beta disease	skos:exactMatch	ORDO:200	Beta rare disease	semapv:LexicalMatching	0.85


MONDO:0009999	gamma disease	skos:exactMatch	ORDO:300	Delta rare disease	semapv:LexicalMatching	0.6
M

ONDO:0009999	gamma disease	skos:exactMatch	ORDO:100	Alpha rare disease	semapv:LexicalMatching	0.3


### The Alignment Puzzle

We have three MONDO diseases and three ORDO diseases to align:

| MONDO | ORDO candidate(s) | Evidence |
|---|---|---|
| MONDO:0001234 (alpha disease) | ORDO:100 (Alpha rare disease) | xref (OBO) + exactMatch (SSSOM, 0.9) |
| MONDO:0005678 (beta disease) | ORDO:200 (Beta rare disease) | xref (OBO) + exactMatch (SSSOM, 0.85) |
| MONDO:0009999 (gamma disease) | ORDO:300 (Delta rare disease) | exactMatch (SSSOM, 0.6) |
| MONDO:0009999 (gamma disease) | ORDO:100 (Alpha rare disease) | exactMatch (SSSOM, 0.3) — conflicting! |

The interesting case is **MONDO:0009999**: it has a moderate match to ORDO:300 and a weaker match to ORDO:100.
But ORDO:100 is already strongly matched to MONDO:0001234, and the ORDO classes are declared **disjoint** in the OWL file.
BOOMER’s reasoner should use these constraints to resolve the ambiguity.

## Step 1: Convert Each Source to a KB

Let’s see what each converter extracts.

In [4]:
%%bash
echo "=== OBO -> YAML ==="
uv run python -m boomer.cli convert /tmp/mondo_subset.obo -o /tmp/mondo.yaml
echo
cat /tmp/mondo.yaml

=== OBO -> YAML ===


Converted /tmp/mondo_subset.obo (obo) to /tmp/mondo.yaml (yaml)


name: mondo-subset
facts:
- fact_type: ProperSubClassOf
  sub: MONDO:0001234
  sup: MONDO:0000001
- 

fact_type: ProperSubClassOf
  sub: MONDO:0005678
  sup: MONDO:0000001
- fact_type: ProperSubClassOf


  sub: MONDO:0009999
  sup: MONDO:0000001
- fact_type: MemberOfDisjointGroup
  sub: MONDO:0009999
  

...

pe: MemberOfDisjointGroup
  sub: ORDO:100
  group: ORDO
pfacts:
- fact:
    fact_type: EquivalentTo


    sub: MONDO:0001234
    equivalent: ORDO:100
  prob: 0.7
- fact:
    fact_type: EquivalentTo
    

sub: MONDO:0005678
    equivalent: ORDO:200
  prob: 0.7
hypotheses: []
labels:
  MONDO:0000001: dise

ase
  MONDO:0001234: alpha disease
  MONDO:0005678: beta disease
  MONDO:0009999: gamma disease
hype

rparams: []
pfacts_entailed: []


In [5]:
%%bash
echo "=== OWL -> YAML ==="
uv run python -m boomer.cli convert /tmp/ordo_subset.ofn -o /tmp/ordo.yaml
echo
cat /tmp/ordo.yaml

=== OWL -> YAML ===


Converted /tmp/ordo_subset.ofn (owl) to /tmp/ordo.yaml (yaml)


name: http://www.orpha.net/ORDO/ordo-subset
facts:
- fact_type: ProperSubClassOf
  sub: ORDO:100
  s

up: ORDO:999
- fact_type: ProperSubClassOf
  sub: ORDO:200
  sup: ORDO:999
- fact_type: ProperSubCla

ssOf
  sub: ORDO:300
  sup: ORDO:999
- fact_type: DisjointWith
  sub: ORDO:100
  sibling: ORDO:200
-

...

00
  sibling: ORDO:300
- fact_type: MemberOfDisjointGroup
  sub: ORDO:100
  group: ORDO
- fact_type:

 MemberOfDisjointGroup
  sub: ORDO:999
  group: ORDO
- fact_type: MemberOfDisjointGroup
  sub: ORDO:

300
  group: ORDO
- fact_type: MemberOfDisjointGroup
  sub: ORDO:200
  group: ORDO
pfacts: []
hypoth

eses: []
labels:
  ORDO:200: Beta rare disease
  ORDO:100: Alpha rare disease
  ORDO:300: Delta rare

 disease
  ORDO:999: Rare disease grouping
hyperparams: []
pfacts_entailed: []


In [6]:
%%bash
echo "=== SSSOM -> YAML ==="
uv run python -m boomer.cli convert /tmp/mondo_ordo_mappings.sssom.tsv -o /tmp/mappings.yaml
echo
cat /tmp/mappings.yaml

=== SSSOM -> YAML ===


Converted /tmp/mondo_ordo_mappings.sssom.tsv (sssom) to /tmp/mappings.yaml (yaml)


name: https://example.org/mondo-ordo-mappings
description: MONDO to ORDO candidate mappings
facts:
-

 fact_type: MemberOfDisjointGroup
  sub: MONDO:0001234
  group: MONDO
- fact_type: MemberOfDisjointG

roup
  sub: ORDO:100
  group: ORDO
- fact_type: MemberOfDisjointGroup
  sub: MONDO:0005678
  group: 

...

b: 0.85
- fact:
    fact_type: EquivalentTo
    sub: MONDO:0009999
    equivalent: ORDO:300
  prob: 

0.6
- fact:
    fact_type: EquivalentTo
    sub: MONDO:0009999
    equivalent: ORDO:100
  prob: 0.3


hypotheses: []
labels:
  MONDO:0001234: alpha disease
  ORDO:100: Alpha rare disease
  MONDO:0005678

: beta disease
  ORDO:200: Beta rare disease
  MONDO:0009999: gamma disease
  ORDO:300: Delta rare d

isease
hyperparams: []
pfacts_entailed: []


## Step 2: Merge All Three KBs

The `merge` command combines facts and pfacts from multiple sources.
Hard facts (hierarchy, disjointness) stay hard; probabilistic facts accumulate.

In [7]:
%%bash
uv run python -m boomer.cli merge \
  /tmp/mondo_subset.obo \
  /tmp/ordo_subset.ofn \
  /tmp/mondo_ordo_mappings.sssom.tsv \
  -o /tmp/merged_alignment.yaml \
  -n "MONDO-ORDO Alignment" \
  -D "Merged from OBO hierarchy, OWL hierarchy, and SSSOM mappings"

echo
echo "=== Merged KB ==="
cat /tmp/merged_alignment.yaml

Merged 3 files into /tmp/merged_alignment.yaml (yaml)
Final KB contains 25 facts and 6 probabilistic

 facts



=== Merged KB ===


name: MONDO-ORDO Alignment
description: Merged from OBO hierarchy, OWL hierarchy, and SSSOM mappings


facts:
- fact_type: ProperSubClassOf
  sub: MONDO:0001234
  sup: MONDO:0000001
- fact_type: ProperS

...

lentTo
    sub: MONDO:0009999
    equivalent: ORDO:300
  prob: 0.6
- fact:
    fact_type: Equivalent

To
    sub: MONDO:0009999
    equivalent: ORDO:100
  prob: 0.3
hypotheses: []
labels:
  MONDO:000000

1: disease
  MONDO:0001234: alpha disease
  MONDO:0005678: beta disease
  MONDO:0009999: gamma disea

se
  ORDO:200: Beta rare disease
  ORDO:300: Delta rare disease
  ORDO:999: Rare disease grouping
  

ORDO:100: Alpha rare disease
hyperparams: []
pfacts_entailed: []


## Step 3: Solve the Alignment

Now let BOOMER reason over the merged KB.
The reasoner will use the disjointness constraints from OWL,
the hierarchy from OBO, and the mapping probabilities from SSSOM
to find the most probable consistent alignment.

In [8]:
%%bash
uv run python -m boomer.cli solve /tmp/merged_alignment.yaml \
  --timeout 60 \
  -O markdown

Loaded KB with 25 facts and 6 probabilistic facts
Starting search...


Solving KB: MONDO-ORDO Alignment with 6 pfacts; threshold=200
Search completed in 0.0897 seconds
Kno

wledge Base: MONDO-ORDO Alignment
KB num pfacts: 6
Description: Merged from OBO hierarchy, OWL hiera

rchy, and SSSOM mappings

Search Statistics:


  Confidence: 0.5000
  Prior Probability: 1.5744e-01
  Posterior Probability: 0.1111
  Combinations 

...

RDO:100 (Alpha rare disease) :: prior: 0.7 posterior: 0.968219477482972
 * True MONDO:0005678 (beta 

disease) ≡ ORDO:200 (Beta rare disease) :: prior: 0.7 posterior: 0.9571896919792617
 * True MONDO:

0009999 (gamma disease) ≡ ORDO:300 (Delta rare disease) :: prior: 0.6 posterior: 0.597209515096065

5
 * False MONDO:0009999 (gamma disease) ≡ ORDO:100 (Alpha rare disease) :: prior: 0.3 posterior: 

0.004650808173223542



## Interpreting the Results

The solver should confirm:

- **MONDO:0001234 ≡ ORDO:100** — reinforced by both xref and SSSOM evidence
- **MONDO:0005678 ≡ ORDO:200** — reinforced by both xref and SSSOM evidence
- **MONDO:0009999 ≡ ORDO:300** — the moderate SSSOM match wins
- **MONDO:0009999 ≡ ORDO:100 is rejected** — because ORDO:100 is already matched to MONDO:0001234, and ORDO classes are disjoint

This demonstrates how structural constraints from ontologies (disjointness, hierarchy) interact with probabilistic evidence from mappings to produce better alignments than either source alone.

## Step 4: Export and Further Analysis

In [9]:
%%bash
# Export solution as TSV for downstream processing
uv run python -m boomer.cli solve /tmp/merged_alignment.yaml \
  --timeout 60 \
  -O tsv \
  -o /tmp/alignment_solution.tsv \
  --quiet

echo "=== Accepted equivalences ==="
grep "^EquivalentTo.*True" /tmp/alignment_solution.tsv | cut -f1-5

echo
echo "=== Rejected equivalences ==="
grep "^EquivalentTo.*False" /tmp/alignment_solution.tsv | cut -f1-5

Solving KB: MONDO-ORDO Alignment with 6 pfacts; threshold=200


=== Accepted equivalences ===


EquivalentTo	MONDO:0001234	ORDO:100	alpha disease	Alpha rare disease
EquivalentTo	MONDO:0005678	ORDO

:200	beta disease	Beta rare disease
EquivalentTo	MONDO:0001234	ORDO:100	alpha disease	Alpha rare dis

ease
EquivalentTo	MONDO:0005678	ORDO:200	beta disease	Beta rare disease
EquivalentTo	MONDO:0009999	O

RDO:300	gamma disease	Delta rare disease



=== Rejected equivalences ===


EquivalentTo	MONDO:0009999	ORDO:100	gamma disease	Alpha rare disease


In [10]:
# Same workflow via the Python API
from boomer.ontology_converter import obo_to_kb, owl_to_kb
from boomer.sssom_converter import sssom_to_kb
from boomer.search import solve, SearchConfig

# Convert each source
mondo_kb = obo_to_kb("/tmp/mondo_subset.obo")
ordo_kb = owl_to_kb("/tmp/ordo_subset.ofn")
mapping_kb = sssom_to_kb("/tmp/mondo_ordo_mappings.sssom.tsv")

# Merge using KB.extend()
merged = mondo_kb.extend(
    facts=ordo_kb.facts + mapping_kb.facts,
    pfacts=ordo_kb.pfacts + mapping_kb.pfacts,
    labels={**ordo_kb.labels, **mapping_kb.labels},
)
merged.name = "MONDO-ORDO Alignment (Python API)"
merged.normalize()

print(f"Merged KB: {len(merged.facts)} hard facts, {len(merged.pfacts)} probabilistic facts")

# Solve
solution = solve(merged, config=SearchConfig(timeout_seconds=60))

print(f"\nConfidence: {solution.confidence:.3f}")
print(f"Combinations explored: {solution.number_of_combinations}")
print("\nResults:")
for sp in solution.solved_pfacts:
    f = sp.pfact.fact
    if f.fact_type == "EquivalentTo":
        status = "ACCEPTED" if sp.truth_value else "REJECTED"
        print(f"  {status}: {f.sub} \u2261 {f.equivalent}  (prior={sp.pfact.prob:.2f}, posterior={sp.posterior_prob:.3f})")

Merged KB: 25 hard facts, 6 probabilistic facts
Solving KB: MONDO-ORDO Alignment (Python API) with 6 pfacts; threshold=200



Confidence: 0.500


Combinations explored: 37

Results:
  ACCEPTED: MONDO:0001234 ≡ ORDO:100  (prior=0.90, posterior=0.968)
  ACCEPTED: MONDO:0005678 ≡ ORDO:200  (prior=0.85, posterior=0.957)
  ACCEPTED: MONDO:0001234 ≡ ORDO:100  (prior=0.70, posterior=0.968)
  ACCEPTED: MONDO:0005678 ≡ ORDO:200  (prior=0.70, posterior=0.957)
  ACCEPTED: MONDO:0009999 ≡ ORDO:300  (prior=0.60, posterior=0.597)
  REJECTED: MONDO:0009999 ≡ ORDO:100  (prior=0.30, posterior=0.005)


## Summary

This tutorial showed how BOOMER combines multiple data sources for ontology alignment:

| Source | Format | Contribution |
|---|---|---|
| MONDO hierarchy | OBO | SubClassOf structure + xref mappings |
| ORDO hierarchy | OWL | SubClassOf structure + disjointness constraints |
| Cross-ontology mappings | SSSOM | Probabilistic equivalence candidates |

Key commands:
```bash
# Convert any source to YAML
pyboomer convert ontology.obo -o kb.yaml
pyboomer convert ontology.owl -o kb.yaml
pyboomer convert mappings.sssom.tsv -o kb.yaml

# Merge multiple sources
pyboomer merge source1.obo source2.ofn mappings.sssom.tsv -o merged.yaml

# Solve
pyboomer solve merged.yaml -O markdown
```